# From Statistical Targets to Financial Decisions

The equation separates a population target, its sample estimate, and the expected loss of an action:

$$
\theta=\theta(P),\qquad \widehat\theta=T(X_1,\ldots,X_n),\qquad
R_P(a)=\mathbb E_P[L(Y,a)].
$$

Here, $P$ is the population distribution; $\theta(P)$ is the target quantity; $\widehat\theta$ is its estimate. $T$ is the rule that turns observations $X_1,\ldots,X_n$ into an estimate; $n$ is the sample size. $a$ is an action; $Y$ is the outcome; $L(Y,a)$ is its loss; $R_P(a)$ is expected loss under $P.$

In [1]:
import sys
import re as _table_number_re
from IPython.display import HTML as _TableHTML, display as _raw_display


def compact_float_format(number_format=None):
    def format_value(value):
        if number_format is None:
            rendered = str(value)
        elif callable(number_format):
            rendered = str(number_format(value))
        else:
            rendered = number_format % value
        return compact_numeric_token(rendered)
    return format_value


def compact_numeric_token(rendered):
    if not _table_number_re.fullmatch(r"[+-]?(?:[0-9]+(?:\.[0-9]*)?|\.[0-9]+)(?:[eE][+-]?[0-9]+)?", rendered):
        return rendered
    value = float(rendered)
    if value != 0.0 and abs(value) < 5e-13:
        return "0"
    if ("." in rendered or "e" in rendered.lower()) and value.is_integer():
        return str(int(value))
    return rendered


def compact_numeric_text(text):
    pattern = r"(?<![0-9.eE])([+-]?(?:[0-9]+(?:\.[0-9]*)?|\.[0-9]+)(?:[eE][+-]?[0-9]+)?)(?![0-9.eE])"
    return _table_number_re.sub(pattern, lambda match: compact_numeric_token(match.group(0)), text)


def compact_table_latex(latex):
    return compact_numeric_text(latex)


def _compact_table_html(value):
    frame = value.to_frame() if isinstance(value, pd.Series) else value
    html = frame._repr_html_()
    return '<div style="display:flex;justify-content:center;">' + compact_numeric_text(html) + "</div>"


def display(value):
    if isinstance(value, (pd.DataFrame, pd.Series)):
        return _raw_display(_TableHTML(_compact_table_html(value)))
    return _raw_display(value)

from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


ROOT = Path.cwd().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from book_support import ensure_dataset
from urllib.request import urlopen
import hashlib

DATA_URL = "https://data.binance.vision/data/spot/monthly/klines/BTCUSDT/1m/BTCUSDT-1m-2021-01.zip"
DATA_DIR = ROOT / "data" / "public" / "binance"
DATA_PATH = ensure_dataset(ROOT, "binance_btcusdt")
DATA_DIR.mkdir(parents=True, exist_ok=True)
expected_data_hash = "9364A063EFA0604D5B9D2536FB68F922DA5AFA3D8F94A26A527471A1E2F87801"
data_hash = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest().upper()
assert data_hash == expected_data_hash
public_columns = ["Open time", "Open", "High", "Low", "Close", "Volume", "Close time", "Quote volume", "Trades", "Buyer base volume", "Buyer quote volume", "Unused"]
raw = pd.read_csv(DATA_PATH, compression="zip", header=None, names=public_columns, nrows=None)
raw["time"] = pd.to_datetime(raw["Open time"], unit="ms", utc=True).dt.tz_convert(None)
raw = raw.rename(columns={"Close": "close", "Volume": "volume", "Quote volume": "quote_volume", "Trades": "trades"})
raw = raw.sort_values("time").drop_duplicates("time").reset_index(drop=True)
import RiskLabAI

assert RiskLabAI.__version__ == "3.2.0"

from RiskLabAI.data.synthetic_data.simulation import form_true_matrix, simulates_cov_mu

FIGURES = ROOT / "outputs" / "figures" / "01_mathematical_language"
TABLES = ROOT / "outputs" / "tables" / "01_mathematical_language"
FIGURE_DATA = ROOT / "outputs" / "figure_data" / "01_mathematical_language"
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)
FIGURE_DATA.mkdir(parents=True, exist_ok=True)

palette = {
    "background": "#FFFFFF",
    "axes": "#FFFFFF",
    "text": "#1B365D",
    "cyan": "#0072B2",
    "amber": "#C76D00",
    "green": "#00866A",
    "rose": "#B83B3B",
    "grid": "#D9DEE5",
}
plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "figure.facecolor": palette["background"],
    "axes.facecolor": palette["axes"],
    "axes.edgecolor": palette["text"],
    "axes.labelcolor": palette["text"],
    "axes.titlecolor": palette["text"],
    "xtick.color": palette["text"],
    "ytick.color": palette["text"],
    "text.color": palette["text"],
    "grid.color": palette["grid"],
    "grid.linestyle": ":",
    "legend.facecolor": palette["axes"],
    "legend.edgecolor": palette["grid"],
    "savefig.facecolor": palette["background"],
})

## From population targets to sample estimates

The first two expressions define the population mean and covariance, while the third estimates the mean from a sample:

$$
\mu=\mathbb E[X],\qquad
\Sigma=\mathbb E[(X-\mu)(X-\mu)^{\mathsf T}],\qquad
\widehat\mu=\frac{1}{n}\sum_{i=1}^{n}X_i.
$$

Here, $X$ is a random vector; $\mu$ is its population mean; $\Sigma$ is its population covariance matrix. $X_i$ is observation $i$; $n$ is the number of observations; $\widehat\mu$ is the sample mean; $\mathsf T$ denotes transpose.

In [2]:
np.random.seed(20260925)
population_mean, population_covariance = form_true_matrix(
    n_blocks=3,
    block_size=2,
    block_correlation=0.65,
)
np.random.seed(20260926)
sample_mean, sample_covariance = simulates_cov_mu(
    population_mean,
    population_covariance,
    n_obs=240,
    shrink=False,
)
asset_labels = [f"Asset {index}" for index in range(1, population_covariance.shape[0] + 1)]
population_covariance_frame = pd.DataFrame(population_covariance, index=asset_labels, columns=asset_labels)
sample_covariance_frame = pd.DataFrame(sample_covariance, index=asset_labels, columns=asset_labels)
population_covariance_frame.to_csv(FIGURE_DATA / "population_covariance.csv", index_label="Asset")
sample_covariance_frame.to_csv(FIGURE_DATA / "sample_covariance.csv", index_label="Asset")

figure, axes = plt.subplots(1, 2, figsize=(10.5, 4.6), constrained_layout=True)
common_limit = max(population_covariance.max(), sample_covariance.max())
images = []
for axis, matrix, title in zip(
    axes,
    [population_covariance, sample_covariance],
    ["Population Covariance", "Sample Covariance"],
):
    image = axis.imshow(matrix, cmap="magma", vmin=0.0, vmax=common_limit)
    images.append(image)
    axis.set_xticks(range(len(asset_labels)), asset_labels, rotation=45, ha="right")
    axis.set_yticks(range(len(asset_labels)), asset_labels)
    axis.set_title(title)
figure.colorbar(images[-1], ax=axes, shrink=0.78, label="Covariance")
figure.savefig(FIGURES / "covariance_estimand_estimator.png", bbox_inches="tight")
figure.savefig(FIGURES / "covariance_estimand_estimator.pdf", bbox_inches="tight")
display(figure)
plt.close(figure)

<Figure size 1050x460 with 3 Axes>

## Choosing optimal actions from a loss function

Squared loss is minimized by the mean, while absolute loss is minimized by a median:

$$
a_2^*=\arg\min_a\mathbb E[(Y-a)^2]=\mathbb E[Y],
\qquad
a_1^*\in\arg\min_a\mathbb E[|Y-a|]=\operatorname{Median}(Y).
$$

Here, $Y$ is the uncertain outcome; $a$ is a candidate action; $a_2^*$ and $a_1^*$ are optimal actions. $\mathbb E$ denotes expectation; $\arg\min$ selects an action with the smallest expected loss; $\operatorname{Median}(Y)$ is any median of $Y.$

In [3]:
public_close = pd.Series(raw["close"].to_numpy(dtype=float), index=pd.DatetimeIndex(raw["time"]), name="Price")
daily_close = public_close.resample("1D").last()
outcomes = 100.0 * np.log(daily_close).diff().dropna().abs().to_numpy()
candidate_actions = np.linspace(0.0, 1.05 * outcomes.max(), 20001)
squared_loss = np.mean((outcomes[:, None] - candidate_actions[None, :]) ** 2, axis=0)
absolute_loss = np.mean(np.abs(outcomes[:, None] - candidate_actions[None, :]), axis=0)
mean_action = outcomes.mean()
median_action = np.median(outcomes)
loss_curve = pd.DataFrame({
    "Absolute daily log return (%)": candidate_actions,
    "Average squared loss": squared_loss,
    "Average absolute loss": absolute_loss,
    "Sample mean": mean_action,
    "Sample median": median_action,
})
loss_curve.to_csv(FIGURE_DATA / "loss_decisions.csv", index=False)

loss_data_summary = pd.DataFrame({
    "Field": ["Provider", "Archive", "Public access", "Sample", "Daily observations", "Archive SHA256"],
    "Value": ["Binance Data Vision", "BTCUSDT-1m-2021-01.zip", "Direct public download", "January 2021", str(len(outcomes)), data_hash],
})
display(loss_data_summary)

figure, axes = plt.subplots(1, 2, figsize=(10.5, 4.3), constrained_layout=True)
axes[0].plot(candidate_actions, squared_loss, color=palette["cyan"], linewidth=2.1)
axes[0].axvline(mean_action, color=palette["amber"], linewidth=1.7, label="Sample mean")
axes[0].set_xlabel("Absolute daily log return (%)")
axes[0].set_ylabel("Average squared loss")
axes[0].set_title("Squared Loss")
axes[0].legend()
axes[1].plot(candidate_actions, absolute_loss, color=palette["green"], linewidth=2.1)
axes[1].axvline(median_action, color=palette["rose"], linewidth=1.7, label="Sample median")
axes[1].set_xlabel("Absolute daily log return (%)")
axes[1].set_ylabel("Average absolute loss")
axes[1].set_title("Absolute Loss")
axes[1].legend()
figure.savefig(FIGURES / "loss_decisions.png", bbox_inches="tight")
figure.savefig(FIGURES / "loss_decisions.pdf", bbox_inches="tight")
display(figure)
plt.close(figure)

,Field,Value
0,Provider,Binance Data Vision
1,Archive,BTCUSDT-1m-2021-1zip
2,Public access,Direct public download
3,Sample,January 2021
4,Daily observations,30
5,Archive SHA256,9364A063EFA0604D5B9D2536FB68F922DA5AFA3D8F94A2...


<Figure size 1050x430 with 2 Axes>

## Portfolio Decision

The first expression gives fully invested minimum-variance weights, and the second measures the variance of estimated weights:

$$
w^*(\Sigma)=\frac{\Sigma^{-1}\mathbf 1}{\mathbf 1^{\mathsf T}\Sigma^{-1}\mathbf 1},
\qquad
\mathcal R(\widehat w;\Sigma)=\widehat w^{\mathsf T}\Sigma\widehat w.
$$

Here, $\Sigma$ is the asset-return covariance matrix; $\Sigma^{-1}$ is its inverse; $\mathbf 1$ is a vector of ones. $w^*(\Sigma)$ is the population choice; $\widehat w$ is an estimated weight vector; $\mathcal R(\widehat w;\Sigma)$ is its population variance.

In [4]:
def minimum_variance_weights(covariance):
    unit = np.ones(covariance.shape[0])
    solution = np.linalg.solve(covariance, unit)
    return solution / (unit @ solution)

population_weights = minimum_variance_weights(population_covariance)
plug_in_weights = minimum_variance_weights(sample_covariance)
population_risk = population_weights @ population_covariance @ population_weights
plug_in_population_risk = plug_in_weights @ population_covariance @ plug_in_weights

estimation_summary = pd.DataFrame({
    "Object": [
        "Expected return vector",
        "Covariance matrix",
        "Minimum-variance decision",
    ],
    "Population magnitude": [
        np.linalg.norm(population_mean),
        np.linalg.norm(population_covariance, ord="fro"),
        population_risk,
    ],
    "Sample magnitude": [
        np.linalg.norm(sample_mean),
        np.linalg.norm(sample_covariance, ord="fro"),
        plug_in_population_risk,
    ],
    "Estimation or decision error": [
        np.linalg.norm(sample_mean - population_mean),
        np.linalg.norm(sample_covariance - population_covariance, ord="fro"),
        plug_in_population_risk - population_risk,
    ],
})
estimation_summary.to_csv(TABLES / "estimation_decision_summary.csv", index=False)
estimation_latex = compact_table_latex(estimation_summary.to_latex(
    index=False,
    float_format=compact_float_format("%.6f"),
    caption="Population targets, sample estimates, and the induced portfolio decision error.",
    label="tab:estimation-decision-summary",
    position="tbp",
))
estimation_latex = estimation_latex.replace(
    r"\begin{tabular}",
    r"\begin{adjustbox}{max width=\textwidth}\begin{tabular}",
    1,
).replace(r"\end{tabular}", r"\end{tabular}\end{adjustbox}", 1)
(TABLES / "estimation_decision_summary.tex").write_text(estimation_latex, encoding="utf-8")
display(estimation_summary.round(6))

,Object,Population magnitude,Sample magnitude,Estimation or decision error
0,Expected return vector,0.427722,0.430952,0.019529
1,Covariance matrix,0.043389,0.043023,0.005568
2,Minimum-variance decision,0.001128,0.001211,0.000083


## Tests

In [5]:
assert population_mean.shape == sample_mean.shape == (6, 1)
assert population_covariance.shape == sample_covariance.shape == (6, 6)
assert np.allclose(population_covariance, population_covariance.T)
assert np.allclose(sample_covariance, sample_covariance.T)
assert np.linalg.eigvalsh(population_covariance).min() > 0.0
assert np.linalg.eigvalsh(sample_covariance).min() > 0.0
assert np.isclose(population_weights.sum(), 1.0)
assert np.isclose(plug_in_weights.sum(), 1.0)
assert plug_in_population_risk >= population_risk - 1.0e-12
assert data_hash == "9364A063EFA0604D5B9D2536FB68F922DA5AFA3D8F94A26A527471A1E2F87801"
assert len(outcomes) == 30
assert abs(candidate_actions[np.argmin(squared_loss)] - mean_action) <= np.diff(candidate_actions).max()
assert np.isclose(absolute_loss.min(), np.mean(np.abs(outcomes - median_action)), rtol=1.0e-8, atol=1.0e-10)
assert len(estimation_summary) == 3